# 11 — The Kuramoto dyad and the ground truth

The capstone begins here. Before we can ask whether a *quantum* coupling measure earns its keep, we need a problem where we already know the answer — a synthetic system whose coupling we *injected ourselves*, so "did the measure recover it?" becomes a question with a checkable answer. This notebook builds that test bed: two coupled **Kuramoto oscillators** with a known coupling strength $K$, and the two **classical baselines** every later measure must beat.

**Prerequisites:** `10_embedding_signals_into_states`.

**What you'll be able to do**
- Write down the **Kuramoto dyad** as a stochastic differential equation and say what its one knob — the coupling strength $K$ — controls.
- Explain why a *synthetic* system is the right starting point: because we set $K$, we hold the **ground truth**, so any measure can be scored against it.
- Simulate the dyad at several $K$ with `simulate_kuramoto_dyad` and read its behaviour off the **phase-difference** trajectory $\theta_1 - \theta_2$ — drifting when uncoupled, locking as $K$ grows.
- Compute the two classical baselines — the **phase-locking value (PLV)** and the **cosine correlation** — and see them climb from near zero toward one. These are the yardsticks the rest of the capstone has to clear.

In `10` you assembled the embedding menu — the bridge from a signal to a quantum state — and proved, on a clean toy, that the choice of bridge decides what the geometry can see. Today you build the *signal*: a real synthetic system with a coupling you control, and the classical numbers that already read that coupling well. That gives the capstone its spine — a ground truth to recover and a bar to clear — and everything from `12` onward is measured against what you set up here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.capstone import (
    simulate_kuramoto_dyad,
    plv,
    cosine_correlation,
)

np.random.seed(42)
viz.use_course_style()

## 1. The synthetic Kuramoto dyad

A Kuramoto oscillator is the simplest honest model of a rhythm: a single phase $\theta(t)$ that advances at its own natural frequency $\omega$ and is nudged by whatever it is coupled to (Kuramoto 1984). Two of them, coupled to each other and stirred by noise, give the smallest system that can *synchronise* — a **dyad**. We write it as a pair of stochastic differential equations:

$$
\mathrm{d}\theta_1 = \bigl(\omega_1 + K\,\sin(\theta_2 - \theta_1)\bigr)\,\mathrm{d}t + \sigma\,\mathrm{d}W_1,
$$
$$
\mathrm{d}\theta_2 = \bigl(\omega_2 + K\,\sin(\theta_1 - \theta_2)\bigr)\,\mathrm{d}t + \sigma\,\mathrm{d}W_2.
$$

Read each term in turn. The $\omega_i\,\mathrm{d}t$ term is the oscillator's own pace — left alone, phase $i$ winds steadily round the circle at rate $\omega_i$. The coupling term $K\,\sin(\theta_j - \theta_i)$ is the pull of the *other* oscillator: the sine is positive when oscillator $j$ is ahead, so the laggard speeds up and the leader is held back — a restoring force toward a common phase, with strength set by the single knob $K$. The $\sigma\,\mathrm{d}W_i$ term is independent white noise on each phase, the realistic jitter that keeps synchrony from ever being perfect. When $K = 0$ the two evolve independently and their phases slide past each other at the rate of their frequency mismatch; as $K$ grows the coupling wins and they **phase-lock**.

Why build a synthetic system at all, rather than reach for recorded data? Because here we *set* $K$ with our own hand. The coupling strength is no longer something to be inferred and argued over — it is a number we chose, the **ground truth**. That is what makes the dyad a measuring bench rather than a mystery: we can hand a measure the simulated phases, ask what coupling it reports, and score its answer against the $K$ we put in. Every measure in this capstone — classical today, quantum from `12` on — will be judged by that one standard: *does it recover the $K$ we injected?*

## 2. Simulate the dyad across coupling strengths

We simulate the dyad at four coupling strengths — $K = 0,\ 0.5,\ 2,\ 4$ — holding the natural frequencies ($\omega_1 = 1.0$, $\omega_2 = 1.2$), the noise level, and the random seed fixed, so the *only* thing changing across the four runs is the coupling. `simulate_kuramoto_dyad` integrates the SDE above by the Euler–Maruyama scheme and returns the two phase traces $\theta_1, \theta_2$.

The cleanest window onto synchrony is not either phase alone — both wind round the circle — but their **difference** $\theta_1 - \theta_2$. If the two drift independently, the difference grows without bound (the phases keep slipping past each other); if they lock, the difference settles to a roughly constant offset. So we plot $\theta_1 - \theta_2$ over time, one line per $K$, and let the shape of each line tell us whether the dyad has locked.

In [ ]:
duration = 200.0
omega_1, omega_2 = 1.0, 1.2
dt = 0.05
K_values = [0.0, 0.5, 2.0, 4.0]

# Same frequencies, same noise, same seed across runs -> only K changes.
trajectories = {
    K: simulate_kuramoto_dyad(
        K=K, omega_1=omega_1, omega_2=omega_2, duration=duration, dt=dt, seed=0
    )
    for K in K_values
}

time = np.arange(int(duration / dt)) * dt
phase_colors = [COLORS["muted"], COLORS["flow"], COLORS["source"], COLORS["highlight"]]

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# (a) The phase difference theta_1 - theta_2: drifts when uncoupled, flattens when locked.
ax = axes[0]
for K, color in zip(K_values, phase_colors):
    t1, t2 = trajectories[K]
    ax.plot(time, t1 - t2, color=color, lw=1.8, label=rf"$K = {K}$")
ax.axhline(0.0, color=COLORS["grid"], lw=1.0, zorder=0)
ax.set_xlabel("time")
ax.set_ylabel(r"phase difference  $\theta_1 - \theta_2$")
ax.set_title("Uncoupled phases drift apart; coupling locks the difference", pad=10)
ax.legend(loc="lower left", ncol=4)

# (b) The two raw rhythms sin(theta) at K = 0 (drifting) vs K = 4 (locked), for context.
ax = axes[1]
t1_free, t2_free = trajectories[0.0]
t1_lock, t2_lock = trajectories[4.0]
window = time < 40.0  # a short window so the individual cycles are legible
ax.plot(time[window], np.sin(t1_free[window]), color=COLORS["source"], lw=1.6,
        label=r"$\sin\theta_1$,  $K = 0$")
ax.plot(time[window], np.sin(t2_free[window]), color=COLORS["target"], lw=1.6,
        label=r"$\sin\theta_2$,  $K = 0$", alpha=0.9)
ax.plot(time[window], np.sin(t1_lock[window]), color=COLORS["quantum"], lw=1.8, ls="--",
        label=r"$\sin\theta_1$,  $K = 4$")
ax.plot(time[window], np.sin(t2_lock[window]), color=COLORS["highlight"], lw=1.8, ls="--",
        label=r"$\sin\theta_2$,  $K = 4$", alpha=0.9)
ax.set_xlabel("time")
ax.set_ylabel(r"$\sin\theta$")
ax.set_title("The raw rhythms: independent at $K = 0$, overlapping at $K = 4$", pad=10)
ax.legend(loc="upper right", ncol=2, fontsize=9)

fig.tight_layout()
plt.show()

**Read the figure.** Two panels, both telling the same story from different angles.

The *top panel* is the phase difference $\theta_1 - \theta_2$, one line per coupling strength. The grey $K = 0$ line slides steadily away from zero: with no coupling, the faster oscillator pulls ahead and the difference *drifts* without settling — the signature of two rhythms that ignore each other. The moment coupling switches on, that drift collapses. The $K = 0.5$ line (sage) already stops drifting and hovers around a roughly constant offset; raising the coupling to $K = 2$ (periwinkle) and $K = 4$ (rose) pins the difference ever more tightly, the band of wiggle around its offset shrinking as the coupling grows. Drift versus a settled, tightening offset — that is what "uncoupled" and "phase-locked" look like.

The *bottom panel* shows the underlying rhythms $\sin\theta$ over a short window, so you can see the locking in the waveforms themselves. At $K = 0$ (solid) the two traces wander out of step — one slowly laps the other. At $K = 4$ (dashed) they rise and fall together, a small fixed phase offset apart, exactly as the flat top-panel line predicted. The noise is still present in both cases — synchrony here is statistical, never perfect — which is precisely the realistic setting the coupling measures will have to cope with.

## 3. The classical baselines — PLV and cosine correlation

A figure shows us synchrony; a *measure* turns it into one number we can compare. Two classical measures are the standard against which any new coupling statistic is judged, and the capstone adopts both as its baselines.

The **phase-locking value** (PLV) is the workhorse of phase synchrony in neuroscience (Lachaux et al. 1999). It asks how *consistent* the phase difference is, ignoring its size:

$$
\mathrm{PLV} = \bigl\lvert \langle e^{i(\theta_1 - \theta_2)} \rangle \bigr\rvert .
$$

Each sample contributes a unit vector pointing at angle $\theta_1 - \theta_2$; PLV is the length of their average. If the difference is scattered all over the circle (no locking), the unit vectors cancel and the average length is near $0$; if the difference is pinned to one value (locked), they all point the same way and the length approaches $1$. PLV is exactly the magnitude of the **first circular moment** of the phase difference — the same quantity `10` showed the naive phase qubit encodes, which is the thread we will pull on in `12`.

The **cosine correlation** is a complementary, Euclidean-flavoured baseline: the ordinary Pearson correlation between the two real signals $\cos\theta_1$ and $\cos\theta_2$. Where PLV lives on the circle and sees only relative phase, the cosine correlation treats the rhythms as plain time series and asks whether they rise and fall together. The two measure overlapping but not identical things, which is why it is worth carrying both.

We evaluate both at the four coupling strengths we already simulated.

In [ ]:
print(f"{'K':>5}  {'PLV':>8}  {'cosine corr':>12}")
print("-" * 29)
plv_values = []
corr_values = []
for K in K_values:
    t1, t2 = trajectories[K]
    plv_K = plv(t1, t2)
    corr_K = abs(cosine_correlation(t1, t2))
    plv_values.append(plv_K)
    corr_values.append(corr_K)
    print(f"{K:>5.1f}  {plv_K:>8.3f}  {corr_K:>12.3f}")

**Read the output.** Both baselines do exactly what a coupling measure should: they sit near zero when the dyad is uncoupled and climb toward one as the coupling takes hold. At $K = 0$ the PLV is small — a residual value, not exactly zero, because a finite noisy record never averages perfectly to nothing — and it jumps to well above $0.9$ the moment coupling is present, rising further toward one as $K$ grows. The cosine correlation tracks the same ascent from a near-zero floor toward one. With these natural frequencies so close together ($\omega_1 = 1.0$, $\omega_2 = 1.2$), the dyad locks readily, so the climb is steep: most of the action happens between $K = 0$ and $K = 0.5$, and the higher couplings mostly tighten an already-locked dyad — which is why their PLV values are close. (The "Your turn" exercises invite you to scan more finely and find exactly where PLV crosses one half.)

These two numbers are the yardsticks. Any quantum coupling measure introduced later has to be read against *this* table: a statistic that merely rises with $K$ has done nothing new, because PLV and cosine correlation already do that. The interesting question — the one the capstone is built to answer honestly — is whether a quantum measure can ever separate situations these baselines cannot. Holding that bar in view is the whole point of writing it down now.

## Your turn

**Warm-up.** Re-run the simulation with a larger noise level by passing `noise_std` to `simulate_kuramoto_dyad` (its default is the value used above). Before you look, predict what stronger noise does to the PLV at a fixed moderate coupling such as $K = 0.5$: does the phase-locking value go up or down, and why? Then check your prediction against the recomputed number.

**Go further.** The table jumps from a small PLV at $K = 0$ straight to a large one at $K = 0.5$, so the threshold where the dyad starts to lock is hidden between those two values. Simulate on a finer grid of coupling strengths between $0$ and $0.5$ and find, to a coupling step of your choosing, the smallest $K$ at which PLV first rises above one half. Describe in words where that crossing sits relative to the four values used in the figure.

**Challenge.** PLV reads only the phase difference, while the cosine correlation also feels the amplitude of $\cos\theta$. Predict, before computing, whether the two baselines will agree more closely at strong coupling or at weak coupling, and give a reason grounded in what each measure sees. Then compare their values across the four coupling strengths and say whether the gap between them widens or narrows as $K$ grows.

## What you built

- You set up the capstone's **synthetic test bed**: a Kuramoto dyad whose coupling strength $K$ you inject yourself, written as a stochastic differential equation whose every term — natural pace, coupling pull, noise — you can name.
- You established why this is the right foundation: because $K$ is *chosen*, it is the **ground truth**, so every coupling measure can be scored on one question — does it recover the coupling we put in?
- You simulated the dyad across four coupling strengths and read its synchrony off the **phase difference** $\theta_1 - \theta_2$: a drifting line when uncoupled, a settled and tightening offset once coupled.
- You computed the two **classical baselines** — the phase-locking value and the cosine correlation — and watched both rise from near zero toward one. These are the yardsticks the rest of the capstone must beat.

Stand back: you now hold both halves of a fair experiment — a system whose answer you know, and the established numbers that already read it well. That is exactly the setting in which a new measure can be put honestly to the test.

## What's next

In `12_naive_phase_embedding_tracks_K` you will carry these same phases across the bridge from `10` — turning them into a quantum state and asking whether a quantum coupling measure tracks the injected $K$ as faithfully as today's baselines do. That is the first move of the capstone proper: from classical yardsticks to a quantum measure standing beside them.

## References

- Y. Kuramoto, *Chemical Oscillations, Waves, and Turbulence* (Springer, 1984). DOI:10.1007/978-3-642-69689-3
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C

**Previous:** `notebooks/04_QuantumOptimalTransport/10_embedding_signals_into_states.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/12_naive_phase_embedding_tracks_K.ipynb`